# Kaggle Infer Qwen Fixed-Span and Merge Viettel JSON

Notebook này dùng adapter đã upload trên Hugging Face Hub để:

- đọc `*.txt`
- đọc `span_dir` đã có `text/type/position`
- gọi Qwen để dự đoán `assertions` và `candidates`
- ghép lại thành JSON đúng format Viettel

Notebook clone repo trực tiếp và dùng code merge trong repo.

In [ ]:
REPO_URL = 'https://github.com/QuangVoAI/VTAR.git'
REPO_BRANCH = 'luong-ontology-ai-y-khoa'
REPO_NAME = 'VTAR'

!rm -rf /kaggle/working/$REPO_NAME
!git clone --branch "$REPO_BRANCH" "$REPO_URL" "/kaggle/working/$REPO_NAME"
%cd /kaggle/working/$REPO_NAME
!pip install -q -U transformers peft accelerate bitsandbytes
!pip uninstall -y torchao || true

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_DIR = Path(f'/kaggle/working/{REPO_NAME}')
sys.path.insert(0, str(REPO_DIR / 'src'))

from vtr_ai.qwen_fixed_span import (
    QWEN_TARGET_TYPES,
    SYSTEM_PROMPT,
    apply_prediction_to_entity,
    build_fixed_span_user_prompt,
    build_runtime_example,
    parse_qwen_json_response,
)
from vtr_ai.validation import validate_output_directory

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Chọn nguồn input/span.
DATA_SOURCE = 'repo'  # 'repo' hoặc 'kaggle_dataset'

# Nếu dùng ngay dữ liệu trong repo.
REPO_INPUT_DIR = REPO_DIR / 'review_packet_68_100'
REPO_SPAN_DIR = REPO_DIR / 'review_packet_68_100'

# Nếu bạn có input/span riêng trên Kaggle Dataset thì sửa tại đây.
KAGGLE_INPUT_DIR = Path('/kaggle/input/vtr-viettel-input/txt')
KAGGLE_SPAN_DIR = Path('/kaggle/input/vtr-viettel-spans/json')

INPUT_DIR = REPO_INPUT_DIR if DATA_SOURCE == 'repo' else KAGGLE_INPUT_DIR
SPAN_DIR = REPO_SPAN_DIR if DATA_SOURCE == 'repo' else KAGGLE_SPAN_DIR

# Base model + LoRA adapter trên Hugging Face Hub.
BASE_MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_REPO = 'SpringWang08/qwen25-7b-fixed-span-viettel-lora'

OUTPUT_DIR = Path('/kaggle/working/viettel_qwen_fixed_span_output')
CONFIG_PATH = REPO_DIR / 'tmp' / 'config.kaggle.standard.yaml'

CONTEXT_WINDOW = 220
SHORTLIST_SIZE = 10
SHORTLIST_MIN_CONFIDENCE = 0.1
MAX_LENGTH = 1536
MAX_NEW_TOKENS = 96

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(
    '\n'.join([
        'knowledge_base:',
        '  icd10_path: src/vtr_ai/data/icd10_standard.json',
        '  rxnorm_path: src/vtr_ai/data/rxnorm_standard.json',
        '  abbreviations_path: src/vtr_ai/data/vi_abbreviations.json',
        '',
        'matching:',
        '  max_candidates: 3',
        '  min_confidence: 0.55',
        '',
        'rules:',
        '  symptom_window: 80',
        '  assertion_window: 60',
        '',
        'output:',
        '  pretty: true',
        '',
        'ner:',
        '  backend: hybrid',
        '  enable_model_backend: false',
        '  provider: lexical',
        '  checkpoint_path: src/vtr_ai/data/checkpoints/lexical_ner.json',
        '  metadata_path: src/vtr_ai/data/checkpoints/transformers_ner_metadata.json',
        '  fallback_to_rules: true',
        '  min_score: 0.5',
    ]) + '\n',
    encoding='utf-8',
)

assert INPUT_DIR.exists(), f'Missing {INPUT_DIR}'
assert SPAN_DIR.exists(), f'Missing {SPAN_DIR}'
assert CONFIG_PATH.exists(), f'Missing {CONFIG_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('input dir:', INPUT_DIR)
print('span dir:', SPAN_DIR)
print('adapter repo:', ADAPTER_REPO)
print('output dir:', OUTPUT_DIR)
print('cuda available:', torch.cuda.is_available())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

from peft import PeftModel
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print('Loaded base model:', BASE_MODEL_NAME)
print('Loaded adapter:', ADAPTER_REPO)

In [ ]:
def generate_prediction(messages, max_new_tokens=MAX_NEW_TOKENS):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = encoded['input_ids'].shape[-1]
    response = tokenizer.decode(generated[0][prompt_len:], skip_special_tokens=True)
    return response, parse_qwen_json_response(response)

In [ ]:
sample_input_path = sorted(INPUT_DIR.glob('*.txt'))[0]
sample_raw_text = sample_input_path.read_text(encoding='utf-8')
sample_entities = json.loads((SPAN_DIR / f'{sample_input_path.stem}.json').read_text(encoding='utf-8'))
sample_entity = next(entity for entity in sample_entities if entity['type'] in QWEN_TARGET_TYPES)

sample_example = build_runtime_example(
    raw_text=sample_raw_text,
    file_name=sample_input_path.name,
    entity=sample_entity,
    config_path=CONFIG_PATH,
    context_window=CONTEXT_WINDOW,
    shortlist_size=SHORTLIST_SIZE,
    shortlist_min_confidence=SHORTLIST_MIN_CONFIDENCE,
)
sample_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': build_fixed_span_user_prompt(sample_example)},
]
sample_response, sample_prediction = generate_prediction(sample_messages)

print('FILE:', sample_input_path.name)
print('ENTITY:', json.dumps(sample_entity, ensure_ascii=False, indent=2))
print('\nPROMPT:')
print(sample_messages[-1]['content'][:2000])
print('\nRAW RESPONSE:')
print(sample_response)
print('\nPARSED:')
print(json.dumps(sample_prediction, ensure_ascii=False, indent=2))

In [ ]:
input_paths = sorted(INPUT_DIR.glob('*.txt'))
print('total txt files:', len(input_paths))

for idx, input_path in enumerate(input_paths, start=1):
    raw_text = input_path.read_text(encoding='utf-8')
    span_path = SPAN_DIR / f'{input_path.stem}.json'
    entities = json.loads(span_path.read_text(encoding='utf-8'))
    merged_entities = []

    for entity in entities:
        entity_type = str(entity['type'])
        if entity_type not in QWEN_TARGET_TYPES:
            merged_entities.append(apply_prediction_to_entity(entity, {'assertions': [], 'candidates': []}))
            continue

        example = build_runtime_example(
            raw_text=raw_text,
            file_name=input_path.name,
            entity=entity,
            config_path=CONFIG_PATH,
            context_window=CONTEXT_WINDOW,
            shortlist_size=SHORTLIST_SIZE,
            shortlist_min_confidence=SHORTLIST_MIN_CONFIDENCE,
        )
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_fixed_span_user_prompt(example)},
        ]
        _, prediction = generate_prediction(messages)
        merged_entities.append(apply_prediction_to_entity(entity, prediction))

    (OUTPUT_DIR / f'{input_path.stem}.json').write_text(
        json.dumps(merged_entities, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    if idx % 10 == 0 or idx == len(input_paths):
        print(f'processed {idx}/{len(input_paths)}')

In [ ]:
validated = validate_output_directory(INPUT_DIR, OUTPUT_DIR)
print('validated files:', len(validated))
print('sample output file:', validated[0].name)
print((OUTPUT_DIR / validated[0].name).read_text(encoding='utf-8')[:2000])

In [ ]:
%cd /kaggle/working
!zip -r viettel_qwen_fixed_span_output.zip viettel_qwen_fixed_span_output